![logog](https://raw.githubusercontent.com/Pacific-AI-Corp/langtest/main/docs/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pacific-AI-Corp/langtest/blob/main/demo/tutorials/llm_notebooks/dataset-notebooks/HeadQA.ipynb)

**LangTest** is an open-source python library designed to help developers deliver safe and effective Natural Language Processing (NLP) models. Whether you are using **John Snow Labs, Hugging Face, Spacy** models or **OpenAI, Cohere, AI21, Hugging Face Inference API and Azure-OpenAI** based LLMs, it has got you covered. You can test any Named Entity Recognition (NER), Text Classification, fill-mask, Translation model using the library. We also support testing LLMS for Question-Answering, Summarization and text-generation tasks on benchmark datasets. The library supports 100+ out of the box tests. For a complete list of supported test categories, please refer to the [documentation](http://langtest.org/docs/pages/docs/test_categories).


# Getting started with LangTest

In [ ]:
%pip install "langtest[llms]==2.8.0"

# Harness and Its Parameters

The Harness class is a testing class for Natural Language Processing (NLP) and LLM models. It evaluates the performance of a NLP model on a given task using test data and generates a report with test results.Harness can be imported from the LangTest library in the following way.

In [ ]:
#Import Harness from the LangTest library
from langtest import Harness

### Set environment for OpenAI

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"

## HeadQA Dataset

In [3]:
prompt = """
You are a medical expert specializing in clinical reasoning and medical knowledge.

You will be given:
1. A medical multiple-choice question.
2. Five answer options labeled A, B, C, D and E.

Your task is to:
- Select the single best answer based on established medical knowledge.
- Return only the corresponding option letter.

Rules:
- Output exactly one uppercase letter: A, B, C, D or E.
- Do not provide any explanation, reasoning, punctuation, or additional text.

Example:

Question:
What is the most common cause of hypothyroidism in the United States?

Options:
A) Iodine deficiency
B) Hashimoto's thyroiditis
C) Graves' disease
D) Thyroidectomy
E) Secondary hypothyroidism

Output(A, B, C, D or E):
B

Now answer the following question.

Question:
{question}

Options():
{options}

Output(A, B, C, D or E):

"""

In [6]:
harness = Harness(
    task="question-answering",
    model={
        "model": "gpt-5.6-luna", 
        "hub": "openai",
        "type": "chat"
    },
    data={"data_source": "HeadQA",
          "split": "test"},
    config={
        "model_parameters": {
            "user_prompt": prompt
        },
        'tests': {
            'defaults': {
                'min_pass_rate': 0.65
            },
            'robustness': {
                'uppercase': {'min_pass_rate': 0.66},
                'lowercase': {'min_pass_rate': 0.66},
                'add_ocr_typo': {'min_pass_rate': 0.66},
                'dyslexia_word_swap': {'min_pass_rate': 0.60}
            }
        }
    }
)

Skipping download. Path '/home/kalyan/.langtest/datasets/headqa' already exists.
Test Configuration : 
 {
 "model_parameters": {
  "user_prompt": "\nYou are a medical expert specializing in clinical reasoning and medical knowledge.\n\nYou will be given:\n1. A medical multiple-choice question.\n2. Five answer options labeled A, B, C, D and E.\n\nYour task is to:\n- Select the single best answer based on established medical knowledge.\n- Return only the corresponding option letter.\n\nRules:\n- Output exactly one uppercase letter: A, B, C, D or E.\n- Do not provide any explanation, reasoning, punctuation, or additional text.\n\nExample:\n\nQuestion:\nWhat is the most common cause of hypothyroidism in the United States?\n\nOptions:\nA) Iodine deficiency\nB) Hashimoto's thyroiditis\nC) Graves' disease\nD) Thyroidectomy\nE) Secondary hypothyroidism\n\nOutput(A, B, C, D or E):\nB\n\nNow answer the following question.\n\nQuestion:\n{question}\n\nOptions():\n{options}\n\nOutput(A, B, C, D or E

In [9]:
# slice the harness.data 
harness.data = harness.data[:100]  # Use only the first 100 samples for testing

### Generating the test cases.

In [10]:
harness.generate()

Generating testcases...: 100%|██████████| 1/1 [00:00<00:00, 13706.88it/s]


In [11]:
testcases = harness.testcases()
testcases.head()

,category,test_type,original_question,perturbed_question,options
0,robustness,uppercase,Form extracellular fibers with high tensile st...,FORM EXTRACELLULAR FIBERS WITH HIGH TENSILE ST...,A) Fibronectin\nB) Collagen\nC) Integrins\nD) ...
1,robustness,uppercase,The cardiolipin phospholipid is abundant in th...,THE CARDIOLIPIN PHOSPHOLIPID IS ABUNDANT IN TH...,A) Internal mitochondrial\nB) External mitocho...
2,robustness,uppercase,It is NOT a function of the intermediate filam...,IT IS NOT A FUNCTION OF THE INTERMEDIATE FILAM...,A) Provide structural support to the cell.\nB)...
3,robustness,uppercase,The multivesicular bodies are:,THE MULTIVESICULAR BODIES ARE:,A) Peroxisomes\nB) Mitochondria\nC) Polysomes\...
4,robustness,uppercase,They form the myelin sheath of the axons in th...,THEY FORM THE MYELIN SHEATH OF THE AXONS IN TH...,A) Oligodendrocytes\nB) Schwann cells.\nC) Mic...


harness.generate() method automatically generates the test cases (based on the provided configuration)

### Running the tests

In [12]:
harness.run()

Running testcases... : 100%|██████████| 352/352 [09:41<00:00,  1.65s/it]


Called after harness.generate() and is to used to run all the tests.  Returns a pass/fail flag for each test.

### Generated Results

In [13]:
results = harness.generated_results()
results.head()

,category,test_type,original_question,perturbed_question,options,expected_result,actual_result,pass
0,robustness,uppercase,Form extracellular fibers with high tensile st...,FORM EXTRACELLULAR FIBERS WITH HIGH TENSILE ST...,A) Fibronectin\nB) Collagen\nC) Integrins\nD) ...,B,B,True
1,robustness,uppercase,The cardiolipin phospholipid is abundant in th...,THE CARDIOLIPIN PHOSPHOLIPID IS ABUNDANT IN TH...,A) Internal mitochondrial\nB) External mitocho...,A,A,True
2,robustness,uppercase,It is NOT a function of the intermediate filam...,IT IS NOT A FUNCTION OF THE INTERMEDIATE FILAM...,A) Provide structural support to the cell.\nB)...,D,D,True
3,robustness,uppercase,The multivesicular bodies are:,THE MULTIVESICULAR BODIES ARE:,A) Peroxisomes\nB) Mitochondria\nC) Polysomes\...,D,D,True
4,robustness,uppercase,They form the myelin sheath of the axons in th...,THEY FORM THE MYELIN SHEATH OF THE AXONS IN TH...,A) Oligodendrocytes\nB) Schwann cells.\nC) Mic...,B,B,True


This method returns the generated results in the form of a pandas dataframe, which provides a convenient and easy-to-use format for working with the test results. You can use this method to quickly identify the test cases that failed and to determine where fixes are needed.

### Final Results

We can call `.report()` which summarizes the results giving information about pass and fail counts and overall test pass/fail flag.

In [14]:
harness.report()

,category,test_type,fail_count,pass_count,pass_rate,minimum_pass_rate,pass
0,robustness,uppercase,1,99,99%,66%,True
1,robustness,lowercase,2,98,98%,66%,True
2,robustness,add_ocr_typo,1,89,99%,66%,True
3,robustness,dyslexia_word_swap,1,61,98%,60%,True
